# SmartFood · Preparación de datos

Normalizar columnas y texto; generar datos intermedios. Requiere los archivos Parquet de data/raw/.

**Estado:** ejecución completa mediante el maestro el 6 de octubre de 2026; verificación de rutas reejecutada el 7 de octubre de 2026. Consulta el [registro de ejecución](../docs/ejecuciones.md) para el detalle por etapa.

**Archivo:** `01_preparacion_datos.ipynb`. Consulta la [guía de ejecución](../docs/notebooks.md) para conocer entradas, salidas y precauciones.


In [1]:
# Resolver la raíz del proyecto desde Jupyter o desde la carpeta del proyecto.
from pathlib import Path
import os
import sys

def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("smartfood")
if PROJECT_ROOT is None:
    raise RuntimeError("Inicia Jupyter desde la carpeta SmartFood o notebooks.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/intermediate", "data/processed", "data/output", "reports", "models"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Configuracion comun: registrar avisos para el resumen de alertas.
from smartfood.entorno_notebook import configurar_notebook
configurar_notebook()


## ✅ Paso 1: Importar librerias

In [2]:
# Importar librerías
import pandas as pd
from smartfood.limpieza import limpiar_columnas, limpiar_valores_texto

## ✅ Paso 2: Carga de datos

In [3]:
recipes = pd.read_parquet("data/raw/recipes.parquet")
print("Data shape: ", recipes.shape)
reviews = pd.read_parquet("data/raw/reviews.parquet")
print("Data shape: ", reviews.shape)

# Validacion basica de carga: los Parquet deben tener filas y las columnas clave no deben faltar.
assert not recipes.empty, "recipes.parquet se cargo vacio."
assert not reviews.empty, "reviews.parquet se cargo vacio."
assert {"RecipeId"}.issubset(set(recipes.columns)) or {"recipeid"}.issubset(set(c.lower() for c in recipes.columns)), (
    "No se encontro una columna de identificador de receta en recipes."
)
assert {"RecipeId"}.issubset(set(reviews.columns)) or {"recipeid"}.issubset(set(c.lower() for c in reviews.columns)), (
    "No se encontro una columna de identificador de receta en reviews."
)


Data shape:  (522517, 28)


Data shape:  (1401982, 8)


## ✅ Paso 3: Limpieza inicial

Eliminamos columnas que no pertenecen al dataset, ademas, normalizamos los nombres de las columnas. También, procedemos a revisar las columnas del dataset para transformar y estandarizar los datos y así poder revisar efectivamente la duplicidad de estos.

In [4]:
# Llamamos a la función 'limpiar_columnas' para eliminar columnas que no pertenecen al dataset,
# por ejemplo, aquellas columnas cuyo nombre contiene "Unnamed" y para normalizar los nombres de las columnas.
print("Dataset de recetas:")
recipes = limpiar_columnas(recipes, verbose=True)
print("\nDataset de reviews:")
reviews = limpiar_columnas(reviews, verbose=True)

Dataset de recetas:
Columnas originales:
['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions']

Columnas normalizadas:
['recipeid', 'name', 'authorid', 'authorname', 'cooktime', 'preptime', 'totaltime', 'datepublished', 'description', 'images', 'recipecategory', 'keywords', 'recipeingredientquantities', 'recipeingredientparts', 'aggregatedrating', 'reviewcount', 'calories', 'fatcontent', 'saturatedfatcontent', 'cholesterolcontent', 'sodiumcontent', 'carbohydratecontent', 'fibercontent', 'sugarcontent', 'proteincontent', 'recipeservings', 'recipeyield', 'recipeinstructions']

Dataset de

Columnas originales:
['ReviewId', 'RecipeId', 'AuthorId', 'AuthorName', 'Rating', 'Review', 'DateSubmitted', 'DateModified']

Columnas normalizadas:
['reviewid', 'recipeid', 'authorid', 'authorname', 'rating', 'review', 'datesubmitted', 'datemodified']


In [5]:
# Se llama a la función 'limpiar_valores_texto' para normalizar los valores de texto de todas las columnas del DataFrame.
# normalización Unicode, eliminación de caracteres especiales, conversión a minúsculas y eliminación de espacios extra
# a cada valor de tipo cadena en el DataFrame.
recipes = limpiar_valores_texto(recipes, lower=True)
reviews = limpiar_valores_texto(reviews, lower=True)

## ✅ Paso 4: Guardar dataset limpio

In [6]:
# 4. Guardar versiones limpias
recipes.to_parquet("data/intermediate/recipes_limpio.parquet", index=False)
reviews.to_parquet("data/intermediate/reviews_limpio.parquet", index=False)

print("✅ Datasets limpios guardados en data/intermediate")

from smartfood.informe import registrar_etapa
registrar_etapa(PROJECT_ROOT, "preparacion", {"recetas": len(recipes), "reviews": len(reviews), "columnas_recetas": len(recipes.columns), "columnas_reviews": len(reviews.columns)}, f"La preparación guarda {len(recipes):,} recetas y {len(reviews):,} reseñas en data/intermediate. Las cifras describen los datos preparados y no una evaluación de modelos.", "01_preparacion_datos.ipynb")


✅ Datasets limpios guardados en data/intermediate


## Alertas de la ejecución

La tabla agrupa los avisos de Python y conserva su detalle original. Si una celda falla antes de llegar aquí, puedes ejecutar esta celda de resumen para consultar las alertas ya registradas. Los errores y los mensajes de diagnóstico impresos no se incluyen automáticamente.


In [7]:
from smartfood.entorno_notebook import mostrar_alertas
mostrar_alertas(PROJECT_ROOT / "reports" / "alertas" / "01_preparacion_datos.json")
from smartfood.informe import generar_informe
generar_informe(PROJECT_ROOT)


WindowsPath('D:/portafolio/Analisis/Recommenders/SmartFood/reports/informe_final/informe_final.md')